# slv_pagos_historial_estados
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · historial de estados de Pagos |
| Propósito | vista validado_pagos (slv_pagos) → silver.pagos_historial_estados (SCD2 sobre estado): una versión por cada cambio de estado del pago. Por qué SCD2 : el reto pide cuidar que Payments cambia después de creado el pedido. Con el historial se mide el recorrido pendiente → aprobado → reembolsado, el tiempo hasta aprobarse y los rechazos. |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.2 · 2026-10-21 · lee validado_pagos: solo pagos que cumplen las reglas de descarte |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt

from comun import parametros_cdc

aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Tabla(s) silver con AUTO CDC: `pagos_historial_estados`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD2** → una versión por cada cambio de las columnas indicadas, con su vigencia en `__START_AT` / `__END_AT`.

In [ ]:
dlt.create_streaming_table(
    name="pagos_historial_estados",
    comment="Pagos: historial SCD2 del estado (pendiente, aprobado, rechazado, reembolsado). "
           "Vigencia en __START_AT / __END_AT.")
aplicar_cdc(target="pagos_historial_estados", source="validado_pagos",
            **parametros_cdc("id_pago", historial_de=["estado"]))